
<h1 id="Sky-background-subtraction-and-aperture-photometry">Sky background subtraction and aperture photometry</h1><p>In this lecture, we will see how to compute and subtract the local sky background around the star under analysis, and then we will check how to determine a proper <em>aperture</em> to measure the flux of the star.</p>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.colors as colors
import pickle
from astropy.io import fits 


<p>We load one scientific frame and the corresponding estimates of the associated error.</p>


In [ ]:
science_list = np.genfromtxt('./group00_GJ3470_20220328/science/science.list', dtype=str)
science_temp_list = science_list[1100:1110]

science_corrected = pickle.load(open('./group00_GJ3470_20220328/correct/' + science_temp_list[5][:-5] + '_corr.p', 'rb'))
science_corrected_err = pickle.load(open('./group00_GJ3470_20220328/correct/' + science_temp_list[5][:-5] + '_corr_errors.p', 'rb'))


<p>We report here the coordinates of the target measured in the previous step of our analysis. We also prepare the meshgrid arrays and then compute the distance from the centre of our target for each pixel in the frame.</p>


In [ ]:
# Refined  coordinates of the target
x_target_refined = 346
y_target_refined = 31


# Computing again the meshgrid arrays 
ylen, xlen = np.shape(science_corrected)
X_axis = np.arange(0, xlen, 1)
Y_axis = np.arange(0, ylen, 1)
X, Y = np.meshgrid(X_axis, Y_axis)

# 2D array with the distance of each pixel from the target star 
target_distance = np.sqrt((X-x_target_refined)**2 + (Y-y_target_refined)**2)


<p>Below, we define a function to plot a circle on the desired coordinates <code>x_pos, ypos</code> with radius <code>radius</code>, all in pixel units. Optional arguments for the function are: the thickness of the circle expressed in pixel (default <code>thickness=0.5</code>), the label associated with the circle (default <code>label=''</code>, which means no label), the colour of the circle in <a href="https://matplotlib.org/stable/gallery/color/named_colors.html">matplotlib standard named colours</a> (default <code>color='w'</code>, i.e., white circle), transparency of the circle (default <code>alpha=1.</code>, where <a href="https://matplotlib.org/stable/gallery/color/set_alpha.html">alpha</a> can go from 0. from a completely transparent circle to 1. for a full circle).</p>
<p>Note the use of the numpy functions  <a href="https://numpy.org/doc/stable/reference/generated/numpy.linspace.html"><code>numpy.linspace</code></a>, <a href="https://numpy.org/doc/stable/reference/generated/numpy.outer.html"><code>numpy.outer</code></a>,  <a href="https://numpy.org/doc/stable/reference/generated/numpy.ravel.html"><code>numpy.ravel</code></a>. Check the documentation by clicking on the corresponding function to get more information about these functions.</p>


In [ ]:


def make_circle_around_star(x_pos, y_pos, radius, thickness=0.5, label='', color='w', alpha=1.):

    # n = 50: the number of points used to approximate each circumference.
    # radii = [radius, radius+thickness]: the inner and outer radii of the annulus
    n, radii = 50, [radius, radius+thickness]

    # range for angular coordinate to draw the circle
    theta = np.linspace(0, 2*np.pi, n, endpoint=True)

    # outer product between two vectors. 
    # given a with size M and b with size N, the outcome c = np.outer(a, b) will have size MxN 
    # https://numpy.org/doc/1.26/reference/generated/numpy.outer.html
    xs = np.outer(radii, np.cos(theta))
    ys = np.outer(radii, np.sin(theta))

    # The slicing expression:
    # xs[1,::-1]
    #  means:
    #    1: select the second row, corresponding to the outer circumference.
    #    ::-1: reverse the order of its elements.
    # Consequently:
    # - The inner circumference is traversed counterclockwise.
    # - The outer circumference is traversed clockwise.
    # This is a trick used to make an empty circle surrouned by a corona

    xs[1,:] = xs[1,::-1]
    ys[1,:] = ys[1,::-1]

    # numpy.ravel returns a contiguous flatten array
    # https://numpy.org/doc/1.26/reference/generated/numpy.ravel.html
    # matplotlib.pyplot.fill will fill the area enclosed by a curve 
    ax.fill(np.ravel(xs)+x_pos, np.ravel(ys)+y_pos, edgecolor=None, facecolor=color, alpha=alpha, label=label)





<h2 id="Computation-of-the-sky-background">Computation of the sky background</h2><p>To compute the average sky background around our star, we need to define a region of the sensor void of stars or other contaminants. In principle, we could select any region of the frame with these characteristics. However, we should ensure that the sky background level is constant across the frame, which is problematic. A more accessible but still effective approach is to measure the sky background level in a ring, or <em>annulus</em>, around our target. If we centre this annulus on the centroid position of our target, we only need to define its <em>inner radius</em> and <em>outer radius</em> to define it.</p>
<p><strong>Inner radius</strong> Since we want to measure the sky background flux, we must ensure that the selected pixels do not contain flux from the target. In other words, the <em>inner radius</em> must be large enough to include our star completely, and eventually include some background.
In the previous lecture, we defined a radius, called <code>inner_radius</code>, to select the pixels over which to compute the centroid position of the target star. We can still use this radius if we selected this radius by excluding the star's flux completely, as already anticipated during the previous notebook.</p>
<p><strong>Outer radius</strong> The outer radius can be as large as you want, but you must be sure that no light from nearby targets enters the annulus. The outer radius should be large enough so that the annulus will include a sufficient number of pixels to determine the average (or median) sky background robustly.</p>
<p>To ensure that our selection of the <code>inner_radius</code> and <code>outer_radius</code> satisfies these requirements, we can plot the portion of the frame including our target with a lower value for <code>vmax</code>, e.g., using just twice the <code>vmin</code> value, together with two circles representing the inner and outer radii. By doing so, we lose information on the flux distribution near the centre of the star - as all the pixels with flux higher than <code>vmax</code> will have the same colour - but we highlight the flux distribution on the wings of the star where the flux coming from the star is comparable with the sky background.</p>


In [ ]:


# Value for the inner radius
inner_radius = 15
outer_radius = 22
    
from matplotlib import colors 

vmin = np.amin(science_corrected[:,100:400])
vmax = np.amax(science_corrected[:,100:400])

vmax = 2*vmin
print('vmin:  {0:.1f}    vmax: {1:.1f}'.format(vmin, vmax))

fig, ax = plt.subplots(1, figsize=(5,5))
im1 = plt.imshow(science_corrected, cmap=plt.colormaps['magma'], norm=colors.LogNorm(vmin=vmin, vmax=vmax), origin='lower')
plt.colorbar(im1,ax=ax, fraction=0.046, pad=0.04)

# Cut the plot around the target star, with some margin with respect to the inner radius
plt.xlim(x_target_refined-outer_radius*1.2, x_target_refined+outer_radius*1.2)
plt.ylim(y_target_refined-outer_radius*1.2, y_target_refined+outer_radius*1.2)

make_circle_around_star(x_target_refined, y_target_refined, inner_radius, label='inner radius')
make_circle_around_star(x_target_refined, y_target_refined, outer_radius, color='y', label='outer radius')

plt.xlabel(' X [pixels]')
plt.ylabel(' Y [pixels]')
plt.legend(loc='upper left')
plt.show()





<p>To compute the sky background, we select the pixel with a distance from the target (<code>target_distance</code>) between <code>inner_radius</code> and <code>outer_radius</code>. The array <code>annulus_selection</code> is a Boolean array with the same shape as the scientific frame, where every pixel will be <code>True</code> if it satisfies the given condition and <code>False</code> otherwise.</p>
<p>Since Python interprets <code>True</code> as 1.0 and <code>False</code> as 0.0 when converting from Boolean to Real, summing over <code>annulus_selection</code> will give the number of pixels satisfying the condition. Be sure that this value is higher than 150, to compute the mean and/or the median over a statistically significant sample size of pixels.</p>
<p>The average (or median) over the selected pixels gives the average sky background <em>per pixel</em>.</p>


In [ ]:


target_distance = np.sqrt((X-x_target_refined)**2 + (Y-y_target_refined)**2)

annulus_selection = (target_distance > inner_radius) & (target_distance<=outer_radius)

sky_flux_average = np.sum(science_corrected[annulus_selection]) / np.sum(annulus_selection)
sky_flux_median = np.median(science_corrected[annulus_selection])
print('Number of pixels included in the annulus: {0:7.0f}'.format(np.sum(annulus_selection)))
print('Average Sky flux: {0:7.1f} photons/pixel'.format(sky_flux_average)) 
print('Median Sky flux: {0:7.1f} photons/pixel'.format(sky_flux_median)) 





<p><strong>HOMEWORK</strong> <br/>
Compute the associated error to the sky background, remembering that every pixel comes with its associated error. <br/>
Are the <em>median</em> and <em>average</em> consistent within the associated errors? If not, can you identify the cause? <br/>
Based on these results, does using the median or the average make a difference?</p>



<h2 id="Remove-the-sky-background-from-the-measured-flux-of-the-star">Remove the sky background from the measured flux of the star</h2><p>The sky background per pixel determined above is only valid in the region surrounding our target star. <br/>
In principle, we should select the frame region from which we want to remove the sky background and perform the computation only on that portion. This approach would be computationally faster, as you have to perform subtraction (and compute the associated error) on fewer pixels. In the end, the calculation across the entire frame is fast enough that the only difference between the two approaches is the code's readability.</p>
<p><strong>HOMEWORK</strong> <br/>
Compute the error associated with the flux on each pixel after subtracting the sky level.</p>


In [ ]:


# Computation on the full frame
science_sky_corrected = science_corrected - sky_flux_average
#science_sky_corrected_error = ?? . 





<h2 id="Aperture-photometry">Aperture photometry</h2><p>The basic principle of aperture photometry is to sum up the observed flux within a given region called <em>aperture</em>. For our analysis, we will employ a circular aperture, thus defined by the centre of the aperture (corresponding to the centroid of the star) and a radius (the <em>aperture</em>),  but in general you can use any shape according to your specific needs, e.g.,  elliptical or isophotal apertures as employed in <a href="https://sextractor.readthedocs.io/en/latest/index.html">SExtractor</a> for the study of galaxies.</p>
<p>The choice of the aperture will depend on your specific science case. Here, we want to measure the star's flux, so it sounds reasonable to choose an aperture that contains the totality of the star's flux. In reality, you also have to consider the error associated with the flux of a given pixel.
Consider a pixel with stellar flux equal to $100$ photons and sky background flux equal to $2000$ photons. The total flux is $2100$ photons, thus the associated error will be approximately $\sqrt{2100} \simeq 50$. This pixel will contribute $100 \pm 50$ photons to the total flux measurement, providing a negligible improvement.</p>
<p>In conclusion, selecting an aperture that includes 100% of the star's flux is not necessary. Indeed, most of the time, it is nearly impossible to identify such an aperture. In some cases, even an aperture including 80% of the flux may provide the best results.</p>
<p>In the plot below, we report the measured flux at different apertures, divided by the total flux within the largest meaningful aperture, as a function of the aperture radius.</p>
<p><strong>HOMEWORK</strong></p>
<ul>
<li>Identify at least two apertures to perform the photometric analysis of your star. Later on, you will learn how to compare the two lightcurves and select the one with the smaller associated error. You can identify the aperture (on the horizontal axis) by checking the intersection of the horizontal lines (representing different fractions of the total flux) with respect to the measured flux.</li>
<li>What is the error associated with the absolute value of each flux measurement? Remember that here you are just summing the pixel values within the aperture, i.e., you are not performing an average.</li>
</ul>


In [ ]:


# Cumulative distribution of flux as a function of radius
plt.figure()

inner_selection = (target_distance < inner_radius)
total_flux = np.sum(science_sky_corrected[inner_selection])

radius_array = np.arange(0, inner_radius + 1., 0.5)
flux_vs_radius = np.zeros_like(radius_array)


for ii, aperture_radius in enumerate(radius_array):
    aperture_selection = (target_distance < aperture_radius)
    flux_vs_radius[ii] = np.sum(science_sky_corrected[aperture_selection])/total_flux
    
plt.scatter(radius_array, flux_vs_radius, c='C1')
plt.axhline(0.80)
plt.axhline(0.85)
plt.axhline(0.90)
plt.axhline(0.95)
plt.axhline(0.99)

plt.xlabel('Aperture [pixels]')
plt.ylabel('Fractional flux within the aperture')
plt.show()





<h2 id="Extra:-comparing-your-star-with-a-multivariate_normal">Extra: comparing your star with a multivariate_normal</h2><p>You may want to check how much the Point Spread Function of your target is departing from a 2D Gaussian shape. This task may keep you busy for weeks, so we will focus only on a qualitative approach.
The idea behind the next block is to compare the photometry as a function of the aperture radius of two simulated stars with the one measured on the scientific frame. To simulate the star, we use a <em>multivariate normal</em> as provided by the <a href="https://scipy.org/">SciPy</a> package <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.multivariate_normal.html"><code>scipy.stats.multivariate_normal</code></a>, assuming an axisymmetric distribution with respect to the centre of the multivariate normal. Here, the Probability Distribution Function <code>pdf</code> will produce our image. Note the use of the numpy function <a href="https://numpy.org/doc/stable/reference/generated/numpy.dstack.html"><code>numpy.dstack</code></a>.</p>


In [ ]:


from scipy.stats import multivariate_normal

xy_range = np.arange(-inner_radius*1.2, inner_radius*1.2, 0.1)
X_gauss, Y_gauss= np.meshgrid(xy_range, xy_range)

pos = np.dstack((X_gauss, Y_gauss))


gauss_distance =np.sqrt((X_gauss)**2 + (Y_gauss)**2)

plot_range = np.arange(0, inner_radius, 0.1)

# Let's compare two different values for the covariance:

mv_normal_cov05 = multivariate_normal(mean=[0., 0.], cov=5., allow_singular=False)
mv_normal_cov05_pdf = mv_normal_cov05.pdf(pos)
plot_cov05_flux = np.zeros_like(plot_range)

mv_normal_cov10 = multivariate_normal(mean=[0., 0.], cov=10., allow_singular=False)
mv_normal_cov10_pdf = mv_normal_cov10.pdf(pos)
plot_cov10_flux = np.zeros_like(plot_range)

mv_normal_cov20 = multivariate_normal(mean=[0., 0.], cov=20., allow_singular=False)
mv_normal_cov20_pdf = mv_normal_cov20.pdf(pos)
plot_cov20_flux = np.zeros_like(plot_range)


for ii, aperture_radius in enumerate(plot_range):

    pdf_selection = (gauss_distance < aperture_radius)
    plot_cov05_flux[ii] =   np.sum(mv_normal_cov05_pdf[pdf_selection])
    plot_cov10_flux[ii] =   np.sum(mv_normal_cov10_pdf[pdf_selection])
    plot_cov20_flux[ii] =   np.sum(mv_normal_cov20_pdf[pdf_selection])



plt.figure()
plt.scatter(radius_array, flux_vs_radius, c='C1', label='Measurmeents')
plt.plot(plot_range,plot_cov05_flux/plot_cov05_flux[-1] , c='C2', label='Covariance= 5.')
plt.plot(plot_range,plot_cov10_flux/plot_cov10_flux[-1] , c='C3', label='Covariance=10.')
plt.plot(plot_range,plot_cov20_flux/plot_cov20_flux[-1] , c='C4', label='Covariance=20.')

plt.xlabel('Aperture [pixels]')
plt.ylabel('Fractional flux within the aperture')
plt.legend()

plt.show()


